## Import libraries

In [1]:
import pandas as pd
import numpy as np

## Load master dataset

In [2]:
model_df = pd.read_csv("../datasets/transactions_master.csv")

## Check shape

In [3]:
model_df.shape

(1852394, 24)

## Inspect available features

In [4]:
model_df.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud', 'dataset_type'],
      dtype='object')

In [5]:
len(model_df.columns)

24

## Identify Useless / High-Cardinality Columns

In [6]:
high_cardinality_cols = [
    "Unnamed: 0",
    "cc_num",
    "first",
    "last",
    "street",
    "trans_num"
]

model_df[high_cardinality_cols].nunique()

Unnamed: 0    1296675
cc_num            999
first             355
last              486
street            999
trans_num     1852394
dtype: int64

## Create Temporal Features

## 1. Convert datetime column

In [7]:
model_df["trans_date_trans_time"] = pd.to_datetime(model_df["trans_date_trans_time"])

## 2. Extract hour

In [8]:
model_df["transaction_hour"] = model_df["trans_date_trans_time"].dt.hour

## 3. Extract day of week

In [9]:
model_df["transaction_day"] = model_df["trans_date_trans_time"].dt.day_name()

## 4. Extract month

In [10]:
model_df["transaction_month"] = model_df["trans_date_trans_time"].dt.month

## 5. Create weekend flag

In [11]:
model_df["is_weekend"] = (
    model_df["transaction_day"]
    .isin(["Saturday", "Sunday"])
    .astype(int)
)

## 6. Verify created columns

In [12]:
model_df[
    [
        "trans_date_trans_time",
        "transaction_hour",
        "transaction_day",
        "transaction_month",
        "is_weekend"
    ]
].head()

,trans_date_trans_time,transaction_hour,transaction_day,transaction_month,is_weekend
0,2019-01-01 00:00:18,0,Tuesday,1,0
1,2019-01-01 00:00:44,0,Tuesday,1,0
2,2019-01-01 00:00:51,0,Tuesday,1,0
3,2019-01-01 00:01:16,0,Tuesday,1,0
4,2019-01-01 00:03:06,0,Tuesday,1,0


## Create Age Feature

## 1. Create customer age

In [13]:
model_df["customer_age"] = 2026 - pd.to_datetime(model_df["dob"]).dt.year

## 2. Verify feature

In [14]:
model_df[["dob", "customer_age"]].head()

,dob,customer_age
0,1988-03-09,38
1,1978-06-21,48
2,1962-01-19,64
3,1967-01-12,59
4,1986-03-28,40


## 3. Check age statistics

In [15]:
model_df["customer_age"].describe()

count    1.852394e+06
mean     5.271065e+01
std      1.739057e+01
min      2.100000e+01
25%      3.900000e+01
50%      5.100000e+01
75%      6.400000e+01
max      1.020000e+02
Name: customer_age, dtype: float64

## Create Transaction Distance Feature

## 1. Create distance feature

In [16]:
model_df["distance"] = np.sqrt(
    (model_df["lat"] - model_df["merch_lat"])**2 +
    (model_df["long"] - model_df["merch_long"])**2
)

## 2. Verify feature

In [17]:
model_df[["lat", "long", "merch_lat", "merch_long", "distance"]].head()

,lat,long,merch_lat,merch_long,distance
0,36.0788,-81.1781,36.011293,-82.048315,0.872830
1,48.8878,-118.2105,49.159047,-118.186462,0.272310
2,42.1808,-112.2620,43.150704,-112.154481,0.975845
3,46.2306,-112.1138,47.034331,-112.561071,0.919802
4,38.4207,-79.4629,38.674999,-78.632459,0.868505


## 3. Check statistics

In [18]:
model_df["distance"].describe()

count    1.852394e+06
mean     7.656171e-01
std      2.847429e-01
min      2.386629e-04
25%      5.648189e-01
50%      7.982886e-01
75%      9.773826e-01
max      1.413364e+00
Name: distance, dtype: float64

## Create Transaction Amount Groups

## 1. Create amount groups

In [19]:
model_df["amount_group"] = pd.cut(
    model_df["amt"],
    bins = [0, 50, 100, 500, 1000, 5000, 30000],
    labels = [
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High",
        "Extreme"
    ]
)

## 2. Verify feature

In [20]:
model_df[["amt", "amount_group"]].head()

,amt,amount_group
0,4.97,Very Low
1,107.23,Medium
2,220.11,Medium
3,45.00,Very Low
4,41.96,Very Low


## 3. Check distribution

In [21]:
model_df["amount_group"].value_counts()

amount_group
Very Low     961229
Low          556287
Medium       313169
High          16190
Very High      5324
Extreme         195
Name: count, dtype: int64

## Create Night Transaction Flag

## 1. Create night flag

In [22]:
model_df["is_night"] = np.where(
    (model_df["transaction_hour"] >= 22) | (model_df["transaction_hour"] <= 4),
    1,
    0
)

## 2. Verify feature

In [23]:
model_df[["transaction_hour", "is_night"]].head(10)

,transaction_hour,is_night
0,0,1
1,0,1
2,0,1
3,0,1
4,0,1
5,0,1
6,0,1
7,0,1
8,0,1
9,0,1


## 3. Check distribution

In [24]:
model_df["is_night"].value_counts()

is_night
0    1357435
1     494959
Name: count, dtype: int64

## Encode Binary Variables

## 1. Encoding

In [25]:
model_df["gender"] = model_df["gender"].map({
        "F": 0,
        "M": 1
    })

## 2. Verify encoding

In [26]:
model_df["gender"].value_counts()

gender
0    1014749
1     837645
Name: count, dtype: int64

## 3. Check sample

In [27]:
model_df[["gender"]].head()

,gender
0,0
1,0
2,1
3,1
4,1


## One-Hot Encode Low-Cardinality Features

## 1. Encoding

In [28]:
low_cardinality_cols = [
    "category",
    "transaction_hour",
    "amount_group"
]

model_df = pd.get_dummies(
    model_df,
    columns = low_cardinality_cols,
    drop_first = True
)

In [29]:
model_df = pd.get_dummies(
    model_df,
    columns = ["transaction_day"],
    drop_first = True
)

## 2. Check updated shape

In [30]:
model_df.shape

(1852394, 75)

## 3. Inspect new columns

In [31]:
model_df.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'amt',
       'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat',
       'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud', 'dataset_type', 'transaction_month',
       'is_weekend', 'customer_age', 'distance', 'is_night',
       'category_food_dining', 'category_gas_transport',
       'category_grocery_net', 'category_grocery_pos',
       'category_health_fitness', 'category_home', 'category_kids_pets',
       'category_misc_net', 'category_misc_pos', 'category_personal_care',
       'category_shopping_net', 'category_shopping_pos', 'category_travel',
       'transaction_hour_1', 'transaction_hour_2', 'transaction_hour_3',
       'transaction_hour_4', 'transaction_hour_5', 'transaction_hour_6',
       'transaction_hour_7', 'transaction_hour_8', 'transaction_hour_9',
       'transaction_hour_10', 'transaction_hour_11', 'transaction_hour_12',
       '

## Drop Useless / Leakage Columns

## 1. Drop columns

In [32]:
drop_cols = [
    "Unnamed: 0",
    "cc_num",
    "first",
    "last",
    "street",
    "trans_num",
    "merchant",
    "city",
    "state",
    "job",
    "dob",
    "trans_date_trans_time",
    "dataset_type"
]

model_df.drop(columns = drop_cols, inplace = True)

## 2. Check updated shape

In [33]:
model_df.shape

(1852394, 62)

## 3. Inspect remaining columns

In [34]:
model_df.columns

Index(['amt', 'gender', 'zip', 'lat', 'long', 'city_pop', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud', 'transaction_month',
       'is_weekend', 'customer_age', 'distance', 'is_night',
       'category_food_dining', 'category_gas_transport',
       'category_grocery_net', 'category_grocery_pos',
       'category_health_fitness', 'category_home', 'category_kids_pets',
       'category_misc_net', 'category_misc_pos', 'category_personal_care',
       'category_shopping_net', 'category_shopping_pos', 'category_travel',
       'transaction_hour_1', 'transaction_hour_2', 'transaction_hour_3',
       'transaction_hour_4', 'transaction_hour_5', 'transaction_hour_6',
       'transaction_hour_7', 'transaction_hour_8', 'transaction_hour_9',
       'transaction_hour_10', 'transaction_hour_11', 'transaction_hour_12',
       'transaction_hour_13', 'transaction_hour_14', 'transaction_hour_15',
       'transaction_hour_16', 'transaction_hour_17', 'transaction_hour_18',
       'transactio

## Separate Features & Target Variable

## 1. Separating features

In [35]:
X = model_df.drop("is_fraud", axis = 1)

y = model_df["is_fraud"]

## 2. Check shapes

In [36]:
print(X.shape)

(1852394, 61)


In [37]:
print(y.shape)

(1852394,)


## 3. Inspect target distribution

In [38]:
y.value_counts()

is_fraud
0    1842743
1       9651
Name: count, dtype: int64

## 4. Check feature columns

In [39]:
X.columns

Index(['amt', 'gender', 'zip', 'lat', 'long', 'city_pop', 'unix_time',
       'merch_lat', 'merch_long', 'transaction_month', 'is_weekend',
       'customer_age', 'distance', 'is_night', 'category_food_dining',
       'category_gas_transport', 'category_grocery_net',
       'category_grocery_pos', 'category_health_fitness', 'category_home',
       'category_kids_pets', 'category_misc_net', 'category_misc_pos',
       'category_personal_care', 'category_shopping_net',
       'category_shopping_pos', 'category_travel', 'transaction_hour_1',
       'transaction_hour_2', 'transaction_hour_3', 'transaction_hour_4',
       'transaction_hour_5', 'transaction_hour_6', 'transaction_hour_7',
       'transaction_hour_8', 'transaction_hour_9', 'transaction_hour_10',
       'transaction_hour_11', 'transaction_hour_12', 'transaction_hour_13',
       'transaction_hour_14', 'transaction_hour_15', 'transaction_hour_16',
       'transaction_hour_17', 'transaction_hour_18', 'transaction_hour_19',
       

## Train-Test Split

## 1. Import

In [40]:
from sklearn.model_selection import train_test_split

## 2. Splitting

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

## 3. Check shapes

In [42]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(1481915, 61)
(370479, 61)
(1481915,)
(370479,)


## 4. Verify fraud distribution

In [43]:
print(y_train.value_counts(normalize = True) * 100)

print(y_test.value_counts(normalize = True) * 100)

is_fraud
0    99.478985
1     0.521015
Name: proportion, dtype: float64
is_fraud
0    99.479053
1     0.520947
Name: proportion, dtype: float64


## Feature Scaling

## 1. Import

In [44]:
from sklearn.preprocessing import StandardScaler

## 2. Select numerical columns

In [45]:
numerical_cols = [
    "amt",
    "zip",
    "lat",
    "long",
    "city_pop",
    "unix_time",
    "merch_lat",
    "merch_long",
    "customer_age",
    "distance"
]

## 3. Initialize scaler

In [46]:
scaler = StandardScaler()

## 4. Scale training & testing data

In [47]:
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

## 5. Verify scaling

In [48]:
X_train[numerical_cols].describe()

,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,customer_age,distance
count,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06,1.481915e+06
mean,-2.445136e-16,-6.291685e-17,7.641889e-16,3.304165e-15,1.946433e-17,-3.853520e-15,8.813057e-16,8.707285e-17,-1.730141e-16,-1.812007e-15
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-4.405132e-01,-1.769290e+00,-3.648682e+00,-5.482390e+00,-2.936886e-01,-1.830236e+00,-3.820127e+00,-5.550269e+00,-1.823315e+00,-2.686587e+00
25%,-3.853071e-01,-8.402399e-01,-7.628061e-01,-4.774667e-01,-2.913091e-01,-8.608386e-01,-7.439637e-01,-4.846126e-01,-7.884555e-01,-7.052761e-01
50%,-1.440596e-01,-2.436450e-02,1.640888e-01,2.008150e-01,-2.856685e-01,-8.747507e-02,1.628564e-01,2.028668e-01,-9.854920e-02,1.145613e-01
75%,8.308326e-02,8.633282e-01,6.703954e-01,7.314992e-01,-2.263956e-01,8.747282e-01,6.688100e-01,7.254881e-01,6.488492e-01,7.437162e-01
max,1.743626e+02,1.900197e+00,5.549154e+00,1.619999e+00,9.339353e+00,1.640758e+00,5.650132e+00,1.691211e+00,2.833552e+00,2.274147e+00


## Handle Class Imbalance Using SMOTE

In [49]:
#!pip install imbalanced-learn

## 1. Import

In [50]:
from imblearn.over_sampling import SMOTE

## 2. Initialize SMOTE

In [51]:
smote = SMOTE(random_state = 42)

## 3. Apply SMOTE

In [52]:
X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

## 4. Check new shapes

In [53]:
print(X_train_smote.shape)

print(y_train_smote.shape)

(2948388, 61)
(2948388,)


## 5. Verify balanced classes

In [54]:
print(y_train_smote.value_counts())

is_fraud
0    1474194
1    1474194
Name: count, dtype: int64


## Save the processed datasets

In [56]:
X_train_smote.to_csv("../datasets/X_train_smote.csv", index = False)

X_test.to_csv("../datasets/X_test.csv", index = False)

y_train_smote.to_csv("../datasets/y_train_smote.csv", index = False)

y_test.to_csv("../datasets/y_test.csv", index = False)